# Flight Delay Prediction — SkyWest Airlines
**ENSF 444 – Machine Learning Systems | Group 45**

This notebook builds and compares three classification models to predict whether a SkyWest flight
will arrive 15+ minutes late based on pre-departure information.

**Models compared:**
1. Logistic Regression (linear baseline)
2. Random Forest (non-linear ensemble)
3. XGBoost (non-linear boosting)

**Dataset:** U.S. Bureau of Transportation Statistics — SkyWest (OO) flights, January 2024

---

## 1. Imports and Data Loading

**How to run this notebook:**
1. Install dependencies: `pip install -r requirements.txt`
2. Ensure `data/skywest_flights.csv` exists in the repo root
3. Run all cells top to bottom from the `notebooks/` directory

In [ ]:
# ── Standard libraries ──
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ──
import matplotlib.pyplot as plt
import seaborn as sns

# ── Preprocessing ──
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     RandomizedSearchCV, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# use a pipeline that is compatible with imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline

# ── Models ──
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Evaluation ──
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

# ── Class imbalance ──
from imblearn.over_sampling import SMOTE

# ── Plot style ──
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# ── Output directory for saved figures ──
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('All imports loaded successfully.')

In [ ]:
# ── Load the SkyWest dataset ──
# The CSV was pre-filtered from the Kaggle "Flight Delay Dataset 2018-2024"
# to only include flights operated by SkyWest Airlines (carrier code OO).
# Source: https://www.kaggle.com/datasets/shubhamsingh42/flight-delay-dataset-2018-2024

df = pd.read_csv('../data/skywest_flights.csv', low_memory=False)
print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df["FlightDate"].min()} to {df["FlightDate"].max()}')
df.head()

---
## 2. Preprocessing

Based on findings from our EDA (`EDA/eda.ipynb`), we apply the following steps:
1. Drop cancelled/diverted flights (no arrival delay to predict)
2. Engineer the binary target variable (Delayed = ArrDelay >= 15 min)
3. Select features available before departure
4. Extract departure hour from scheduled departure time
5. Group rare airports into an "Other" category to reduce cardinality
6. One-hot encode categorical features (Origin, Dest)
7. Stratified 80/20 train-test split
8. Scale numerical features with StandardScaler
9. Apply SMOTE to handle class imbalance (75.9% on-time vs 24.1% delayed)

In [ ]:
# ── Step 2.1: Remove cancelled/diverted flights ──
# These flights have no ArrDelay value, so they cannot be used for prediction.

print(f'Original rows: {len(df):,}')

# Drop cancelled flights
df = df[df['Cancelled'] == 0.0].copy()
print(f'After removing cancelled flights: {len(df):,}')

# Drop diverted flights
if 'Diverted' in df.columns:
    df = df[df['Diverted'] == 0.0].copy()
    print(f'After removing diverted flights: {len(df):,}')

# Drop any remaining rows with missing ArrDelay
df = df.dropna(subset=['ArrDelay']).copy()
print(f'After removing missing ArrDelay: {len(df):,}')

In [ ]:
# ── Step 2.2: Engineer binary target variable ──
# A flight is considered "Delayed" if arrival delay is 15 minutes or more.
# This threshold aligns with the official BTS definition of a delayed flight.

df['Delayed'] = (df['ArrDelay'] >= 15).astype(int)

# Check class distribution
counts = df['Delayed'].value_counts()
pcts = df['Delayed'].value_counts(normalize=True) * 100

print('Target class distribution:')
print(f'  On-Time (0): {counts[0]:,} ({pcts[0]:.1f}%)')
print(f'  Delayed (1): {counts[1]:,} ({pcts[1]:.1f}%)')
print(f'  Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

In [ ]:
# ── Step 2.3: Select features ──
# We keep only features that are available BEFORE departure and useful for
# prediction. The 120-column dataset has 53 columns that are >=95% null
# (diversion fields, codeshare metadata) - we drop all of those.
#
# Features selected based on EDA findings:
#   - DayOfWeek: delay rates vary by day (Friday highest)
#   - DayofMonth: captures within-month patterns
#   - CRSDepTime: delay rates increase throughout the day
#   - CRSElapsedTime: scheduled flight duration
#   - Distance: flight distance in miles
#   - DepDelay: departure delay (strong predictor, but potential leakage)
#   - Origin, Dest: airport-level delay rate variation (7.7% to 46.5%)

features = ['DayOfWeek', 'DayofMonth', 'CRSDepTime', 'CRSElapsedTime',
            'Distance', 'DepDelay', 'Origin', 'Dest']

target = 'Delayed'

# Keep only selected columns
df_model = df[features + [target]].copy()

# Drop rows with missing DepDelay (small number from cancelled-adjacent records)
print(f'Rows before dropping missing DepDelay: {len(df_model):,}')
df_model = df_model.dropna(subset=['DepDelay']).copy()
print(f'Rows after: {len(df_model):,}')
print(f'\nFeatures selected: {features}')
print(f'Target: {target}')
df_model.head()

In [ ]:
# ── Step 2.4: Feature engineering ──
# Extract departure hour from CRSDepTime (format: HHMM as integer).
# EDA showed delay rates increase throughout the day (cascading delays),
# so hour is more informative than the raw HHMM value.

df_model['DepHour'] = df_model['CRSDepTime'] // 100

# Drop the raw CRSDepTime since we now have DepHour
df_model = df_model.drop(columns=['CRSDepTime'])

print('Added DepHour feature (0-23), dropped raw CRSDepTime.')
print(f'DepHour distribution:\n{df_model["DepHour"].value_counts().sort_index()}')

In [ ]:
# ── Step 2.5: Group rare airports ──
# The dataset has 236 unique origins and 235 unique destinations.
# Many airports have very few flights, which would create sparse one-hot columns.
# We keep airports with >= 100 flights and group the rest as "Other".

MIN_FLIGHTS = 100

# Group rare Origin airports
origin_counts = df_model['Origin'].value_counts()
frequent_origins = origin_counts[origin_counts >= MIN_FLIGHTS].index
df_model['Origin'] = df_model['Origin'].where(
    df_model['Origin'].isin(frequent_origins), 'Other'
)

# Group rare Dest airports
dest_counts = df_model['Dest'].value_counts()
frequent_dests = dest_counts[dest_counts >= MIN_FLIGHTS].index
df_model['Dest'] = df_model['Dest'].where(
    df_model['Dest'].isin(frequent_dests), 'Other'
)

print(f'Origin airports: {df_model["Origin"].nunique()} (rare grouped as "Other")')
print(f'Dest airports: {df_model["Dest"].nunique()} (rare grouped as "Other")')

In [ ]:
# ── Step 2.6: One-hot encode categorical features ──
# Origin and Dest are nominal categorical variables with no inherent order,
# so one-hot encoding is the appropriate transformation.

df_model = pd.get_dummies(df_model, columns=['Origin', 'Dest'], drop_first=True)

print(f'Shape after one-hot encoding: {df_model.shape}')
print(f'Total features: {df_model.shape[1] - 1}')  # minus 1 for target column

In [ ]:
# ── Step 2.7: Train-test split ──
# 80/20 stratified split to preserve class distribution in both sets.

X = df_model.drop(columns=[target])
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Copy to avoid SettingWithCopyWarning when scaling in-place
X_train = X_train.copy()
X_test = X_test.copy()

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')
print(f'\nTraining class distribution:')
print(f'  On-Time: {(y_train == 0).sum():,} ({(y_train == 0).mean()*100:.1f}%)')
print(f'  Delayed: {(y_train == 1).sum():,} ({(y_train == 1).mean()*100:.1f}%)')

In [ ]:
# ── Step 2.8: Note on scaling and SMOTE ──
# Scaling and SMOTE are NOT applied here. They are included as steps inside
# every model pipeline (see Section 3) so they are refit fresh on each
# cross-validation fold.
#
# Why not preprocess the full training set up front?
#   If StandardScaler is fit on the full X_train and SMOTE is run on the full
#   X_train BEFORE GridSearchCV, then when CV later splits that data into
#   folds, the validation portion of each fold has already been "seen" by
#   the scaler (its mean/std were partly computed from those rows) and is
#   contaminated by SMOTE-generated synthetic samples derived from other
#   rows. The CV scores would be inflated.
#
# Solution: include StandardScaler and SMOTE inside an imblearn Pipeline.
#   For each CV fold, the pipeline refits the scaler and SMOTE on only
#   that fold's training portion, then transforms/predicts on the validation
#   portion using those statistics alone. No leakage.
#
# Tree-based models (RF, XGBoost) don't strictly need scaling, but we keep
# it in their pipelines for consistency — it's a no-op for tree splits.

numerical_cols = ['DayOfWeek', 'DayofMonth', 'CRSElapsedTime',
                  'Distance', 'DepDelay', 'DepHour']
print(f'Numerical columns (will be scaled inside pipelines): {numerical_cols}')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

In [ ]:
# ── Step 2.9: Class imbalance summary ──
# We do NOT call SMOTE here. SMOTE will run inside each model pipeline,
# only on the training portion of each CV fold (see explanation in Step 2.8).
# We only print the imbalance ratio so the rest of the notebook has context.

n_on_time = (y_train == 0).sum()
n_delayed = (y_train == 1).sum()
print(f'Training set class distribution (BEFORE in-pipeline SMOTE):')
print(f'  On-Time: {n_on_time:,} ({n_on_time / len(y_train) * 100:.1f}%)')
print(f'  Delayed: {n_delayed:,} ({n_delayed / len(y_train) * 100:.1f}%)')
print(f'  Imbalance ratio: {n_on_time / n_delayed:.1f}:1')
print('\nSMOTE will be applied inside each pipeline during fit().')

---
## 3. Model Training

We train all three models first with default hyperparameters (baseline),
then tune each one. Each team member is responsible for one model.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a trained model on the test set and return a dictionary of metrics.
    
    Parameters:
        model: trained sklearn-compatible classifier
        X_test: test feature matrix
        y_test: test labels
        model_name: string label for display
    
    Returns:
        dict with accuracy, precision, recall, f1, and roc_auc scores
    """
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }
    
    print(f'\n-- {model_name} --')
    print(f'  Accuracy:  {metrics["Accuracy"]:.4f}')
    print(f'  Precision: {metrics["Precision"]:.4f}')
    print(f'  Recall:    {metrics["Recall"]:.4f}')
    print(f'  F1-Score:  {metrics["F1-Score"]:.4f}')
    print(f'  ROC-AUC:   {metrics["ROC-AUC"]:.4f}')
    
    return metrics

### 3.1 Logistic Regression (Linear Baseline)

In [ ]:
# ── 3.1a Logistic Regression — Baseline (Default Hyperparameters) ──
# Serves as our baseline model. Fast to train, highly interpretable,
# and provides probability estimates. Establishes a performance floor
# against which the non-linear models are compared.
#
# Wrapped in an imblearn Pipeline so StandardScaler and SMOTE refit fresh
# inside each CV fold (see Step 2.8).

lr_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_metrics = evaluate_model(lr_pipeline, X_test, y_test, 'Logistic Regression')

In [ ]:
# ── 3.1b Logistic Regression — Hyperparameter Tuning (GridSearchCV) ──
# Tuning the regularization strength (C) and penalty type.
# solver='saga' supports both L1 and L2 penalties.
# Scoring on F1 since accuracy is misleading with imbalanced classes.
# 5-fold stratified cross-validation preserves class distribution in each fold.
#
# Pipeline parameters use the 'lr__' prefix to target the LR step.

lr_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

lr_param_grid = {
    'lr__C': [0.01, 0.1, 1, 10],
    'lr__penalty': ['l1', 'l2'],
    'lr__solver': ['saga']
}

lr_grid = GridSearchCV(
    lr_pipeline,
    param_grid=lr_param_grid,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print(f'Best LR params: {lr_grid.best_params_}')
print(f'Best LR CV F1:  {lr_grid.best_score_:.4f}')

lr_tuned = lr_grid.best_estimator_
lr_tuned_metrics = evaluate_model(lr_tuned, X_test, y_test, 'Logistic Regression (Tuned)')

In [ ]:
# ── 3.1c Logistic Regression — Confusion Matrix ──

fig, ax = plt.subplots(figsize=(6, 5))
y_pred_lr = lr_tuned.predict(X_test)
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(cm_lr, display_labels=['On-Time', 'Delayed'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Logistic Regression (Tuned) — Confusion Matrix')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/lr_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── 3.1d Logistic Regression — Classification Report ──

print('Logistic Regression (Tuned) — Classification Report')
print(classification_report(y_test, y_pred_lr, target_names=['On-Time', 'Delayed']))

### 3.2 Random Forest (Non-linear Ensemble)

TODO

In [ ]:
# ── 3.2a Random Forest — Baseline ──
# TODO: Train Random Forest with default hyperparameters
# rf_model = RandomForestClassifier(...)
# rf_model.fit(X_train_smote, y_train_smote)
# rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

rf_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_metrics = evaluate_model(rf_pipeline, X_test, y_test, 'Random Forest')

In [ ]:
# ── 3.2b Random Forest — Hyperparameter Tuning ──
# Use RandomizedSearchCV to tune RF hyperparameters
# rf_tuned_metrics = evaluate_model(rf_tuned, X_test, y_test, 'Random Forest (Tuned)')


# Random Forest pipeline tuning
rf_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

rf_param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest parameters:", rf_grid.best_params_)

rf_tuned = rf_grid.best_estimator_

rf_tuned_metrics = evaluate_model(rf_tuned, X_test, y_test, 'Random Forest (Tuned)')

In [ ]:
# Confusion Matrix - Random Forest Pipeline
y_pred_rf = rf_tuned.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['On-Time', 'Delayed'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Random Forest Confusion Matrix')
plt.tight_layout()

fig.savefig(f'{RESULTS_DIR}/rf_confusion_matrix.png', dpi=150)
plt.show()

### 3.3 XGBoost (Non-linear Boosting)

XGBoost is a gradient-boosted decision tree ensemble. Each successive tree
is trained to correct the errors of the previous trees, producing a strong
non-linear model that typically outperforms a single decision tree or a
random forest on tabular data.

**Why XGBoost for this problem:**
- Handles non-linear feature interactions (e.g., late-evening flights from
  certain airports being especially delay-prone) without manual interaction
  features.
- Robust to feature scale (we still keep `StandardScaler` in the pipeline
  for consistency with the other models — it's a no-op for tree splits).
- Has well-developed regularization (`reg_alpha`, `reg_lambda`,
  `min_child_weight`) which helps avoid overfitting on the smaller
  delayed-class samples.

**Tuning approach:** The full XGBoost grid would have hundreds of
combinations, so we use `RandomizedSearchCV` with `n_iter=20` to sample
20 random combinations — much faster than full grid search, and usually
finds a near-optimal configuration.

In [ ]:
# ── 3.3a XGBoost — Baseline (Default Hyperparameters) ──
# Wrapped in an imblearn Pipeline so StandardScaler and SMOTE refit fresh
# inside each CV fold (see Step 2.8).
#
# Notes on the constructor args:
#   eval_metric='logloss' — silences a deprecation warning in xgboost >=1.3
#   use_label_encoder=False is no longer needed in xgboost >=2.0 so omitted
#   tree_method='hist' — faster histogram-based splitting (CPU)
#   n_jobs=-1 — parallel tree building across cores

xgb_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(
        eval_metric='logloss',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)
xgb_metrics = evaluate_model(xgb_pipeline, X_test, y_test, 'XGBoost')

In [ ]:
# ── 3.3b XGBoost — Hyperparameter Tuning (RandomizedSearchCV) ──
# RandomizedSearchCV samples n_iter random combinations from the grid
# instead of trying all of them. With 5 hyperparameters that each have
# multiple values, full grid search would be hundreds of fits; n_iter=20
# keeps tuning tractable while still exploring most of the space.
#
# Hyperparameters being tuned:
#   n_estimators     — number of boosting rounds (more = stronger but slower)
#   max_depth        — max depth of each tree (controls model complexity)
#   learning_rate    — shrinkage applied to each tree's contribution
#   subsample        — row sub-sampling per tree (regularization)
#   colsample_bytree — column sub-sampling per tree (regularization)

xgb_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(
        eval_metric='logloss',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_dist = {
    'xgb__n_estimators':     [100, 200, 300],
    'xgb__max_depth':        [3, 5, 7, 9],
    'xgb__learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'xgb__subsample':        [0.7, 0.85, 1.0],
    'xgb__colsample_bytree': [0.7, 0.85, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    random_state=42
)

xgb_search.fit(X_train, y_train)

print(f'Best XGBoost params: {xgb_search.best_params_}')
print(f'Best XGBoost CV F1:  {xgb_search.best_score_:.4f}')

xgb_tuned = xgb_search.best_estimator_
xgb_tuned_metrics = evaluate_model(xgb_tuned, X_test, y_test, 'XGBoost (Tuned)')

In [ ]:
# ── 3.3c XGBoost — Confusion Matrix & Classification Report ──

y_pred_xgb = xgb_tuned.predict(X_test)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['On-Time', 'Delayed'])
disp.plot(cmap='Blues', ax=ax, colorbar=False, values_format='d')
ax.set_title('XGBoost (Tuned) — Confusion Matrix')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/xgb_confusion_matrix.png', dpi=150)
plt.show()

print('\nXGBoost (Tuned) — Classification Report')
print(classification_report(y_test, y_pred_xgb, target_names=['On-Time', 'Delayed']))

---
## 4. Model Comparison

*TODO — combine all tuned model results into comparison table, ROC curves, and metrics bar chart once all three models are complete.*

In [ ]:
# ── 4.1 Final Comparison Table ──
# TODO: Uncomment once all three models are trained
# tuned_results = pd.DataFrame([lr_tuned_metrics, rf_tuned_metrics, xgb_tuned_metrics])
# tuned_results = tuned_results.set_index('Model')
# tuned_results.round(4)

In [ ]:
# ── 4.2 ROC Curves ──
# TODO: Plot ROC curves for all three tuned models on one figure

In [ ]:
# ── 4.3 Metrics Bar Chart ──
# TODO: Side-by-side bar chart comparing all metrics across models

---
## 5. DepDelay Leakage Experiment

*TODO — retrain all three models WITHOUT DepDelay to assess data leakage.*

EDA showed DepDelay correlates 0.98 with ArrDelay. This experiment compares
model performance with and without DepDelay to quantify the leakage impact.

In [ ]:
# ── 5.1 Rebuild features without DepDelay ──
# TODO

In [ ]:
# ── 5.2 Train models without DepDelay ──
# TODO

In [ ]:
# ── 5.3 Compare with vs without DepDelay ──
# TODO

---
## 6. Feature Importance and Final Summary

*TODO — feature importance plots and final recommendation.*

In [ ]:
# ── 6.1 Feature Importance ──
# TODO

In [ ]:
# ── 6.2 Final Summary ──
# TODO